In [1]:
import tensorflow as tf
import numpy as np

IMG_SIZE = (224, 224)
IMG_SHAPE = IMG_SIZE + (3,)

# Load only to read its weights and confirm class count — not to reuse any of its layers/graph
trained_model = tf.keras.models.load_model("../models/mobilenetv2_finetuned.keras")
NUM_CLASSES = trained_model.output_shape[-1]
print(f"Detected {NUM_CLASSES} output classes")

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
], name="data_augmentation")

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SHAPE, include_top=False, weights=None  # None, not "imagenet" — no point downloading pretrained weights we're about to overwrite anyway
)
base_model.trainable = True  # matches the fine-tuned state's topology; harmless for inference either way since training=False is hardcoded below

inputs = tf.keras.Input(shape=IMG_SHAPE)
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax")(x)
rebuilt_model = tf.keras.Model(inputs, outputs)

rebuilt_model.set_weights(trained_model.get_weights())
print("Weights transplanted.")

Detected 38 output classes
Weights transplanted.


In [2]:
sample_input = tf.random.uniform((4,) + IMG_SHAPE, minval=0, maxval=255)

pred_original = trained_model.predict(sample_input)
pred_rebuilt = rebuilt_model.predict(sample_input)

max_diff = np.max(np.abs(pred_original - pred_rebuilt))
print(f"Max abs diff: {max_diff:.6f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Max abs diff: 0.000000


In [3]:
export_path = "../models/classifier_savedmodel_v2"
rebuilt_model.export(export_path)
print(f"Exported to {export_path}")

INFO:tensorflow:Assets written to: ../models/classifier_savedmodel_v2\assets


INFO:tensorflow:Assets written to: ../models/classifier_savedmodel_v2\assets


Saved artifact at '../models/classifier_savedmodel_v2'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_490')
Output Type:
  TensorSpec(shape=(None, 38), dtype=tf.float32, name=None)
Captures:
  1571761258576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1571761274704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1571761569872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1571761570064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1571761571408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1571761571216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1571761570448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1571761571984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1571761570640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1571761571600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  15717615717

In [4]:
base_model_infer = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SHAPE, include_top=False, weights=None
)
base_model_infer.trainable = True

inputs = tf.keras.Input(shape=IMG_SHAPE)
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)  # augmentation layer simply skipped — never called
x = base_model_infer(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax")(x)
inference_model = tf.keras.Model(inputs, outputs)

inference_model.set_weights(trained_model.get_weights())

In [5]:
pred_original = trained_model.predict(sample_input)
pred_inference = inference_model.predict(sample_input)

max_diff = np.max(np.abs(pred_original - pred_inference))
print(f"Max abs diff: {max_diff:.6f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 618ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
Max abs diff: 0.000000


In [6]:
export_path_v3 = "../models/classifier_savedmodel_v3"
inference_model.export(export_path_v3)

INFO:tensorflow:Assets written to: ../models/classifier_savedmodel_v3\assets


INFO:tensorflow:Assets written to: ../models/classifier_savedmodel_v3\assets


Saved artifact at '../models/classifier_savedmodel_v3'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_656')
Output Type:
  TensorSpec(shape=(None, 38), dtype=tf.float32, name=None)
Captures:
  1571708982032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1571780811216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1571780810832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1571780807376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1571780820816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1571780807568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1571780810064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1571780804688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1571780811408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1571780819856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  15717808204

In [8]:
def dice_loss(y_true, y_pred, smooth=1e-6):
    axes = [1, 2, 3]
    intersection = tf.reduce_sum(y_true * y_pred, axis=axes)
    union = tf.reduce_sum(y_true, axis=axes) + tf.reduce_sum(y_pred, axis=axes)
    dice = (2. * intersection + smooth) / (union + smooth)
    return 1 - tf.reduce_mean(dice)

def bce_dice_loss(y_true, y_pred):
    bce = tf.reduce_mean(tf.keras.losses.binary_crossentropy(y_true, y_pred))
    return bce + dice_loss(y_true, y_pred)

def iou_metric(y_true, y_pred, threshold=0.5, smooth=1e-6):
    y_pred_bin = tf.cast(y_pred > threshold, tf.float32)
    axes = [1, 2, 3]
    intersection = tf.reduce_sum(y_true * y_pred_bin, axis=axes)
    union = tf.reduce_sum(y_true, axis=axes) + tf.reduce_sum(y_pred_bin, axis=axes) - intersection
    iou = (intersection + smooth) / (union + smooth)
    return tf.reduce_mean(iou)

In [9]:
def conv_block(x, filters, dropout_rate=0.0):
    x = tf.keras.layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    if dropout_rate > 0:
        x = tf.keras.layers.Dropout(dropout_rate)(x)
    return x

def build_unet_mobilenet(input_shape=(128, 128, 3)):
    base_model = tf.keras.applications.MobileNetV2(input_shape=input_shape, include_top=False)

    skip_names = [
        'block_1_expand_relu', 'block_3_expand_relu', 'block_6_expand_relu',
        'block_13_expand_relu', 'block_16_project',
    ]
    skip_outputs = [base_model.get_layer(name).output for name in skip_names]

    down_stack = tf.keras.Model(inputs=base_model.input, outputs=skip_outputs)
    down_stack.trainable = False

    inputs = tf.keras.Input(shape=input_shape)
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
    s1, s2, s3, s4, bottleneck = down_stack(x)

    def upsample_block(x, skip, filters):
        x = tf.keras.layers.Conv2DTranspose(filters, 3, strides=2, padding="same")(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.ReLU()(x)
        x = tf.keras.layers.Concatenate()([x, skip])
        x = tf.keras.layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = tf.keras.layers.BatchNormalization()(x)
        return x

    x = upsample_block(bottleneck, s4, 512)
    x = upsample_block(x, s3, 256)
    x = upsample_block(x, s2, 128)
    x = upsample_block(x, s1, 64)

    x = tf.keras.layers.Conv2DTranspose(32, 3, strides=2, padding="same")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)

    outputs = tf.keras.layers.Conv2D(1, 1, activation="sigmoid")(x)
    return tf.keras.Model(inputs, outputs, name="unet_mobilenet_severity")

In [10]:
trained_unet = tf.keras.models.load_model(
    "../models/unet_mobilenet_finetuned.keras",
    custom_objects={"bce_dice_loss": bce_dice_loss, "iou_metric": iou_metric}
)

# Rebuild the exact same architecture fresh — function already defined earlier in this notebook's session
inference_unet = build_unet_mobilenet(input_shape=(128, 128, 3))
inference_unet.set_weights(trained_unet.get_weights())
print("U-Net weights transplanted.")

U-Net weights transplanted.


In [11]:
sample_seg_input = tf.random.uniform((2, 128, 128, 3), minval=0, maxval=1)

pred_original = trained_unet.predict(sample_seg_input)
pred_inference = inference_unet.predict(sample_seg_input)

max_diff = np.max(np.abs(pred_original - pred_inference))
print(f"Max abs diff: {max_diff:.6f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Max abs diff: 0.000000


In [12]:
export_path_unet = "../models/severity_unet_savedmodel_v2"
inference_unet.export(export_path_unet)

INFO:tensorflow:Assets written to: ../models/severity_unet_savedmodel_v2\assets


INFO:tensorflow:Assets written to: ../models/severity_unet_savedmodel_v2\assets


Saved artifact at '../models/severity_unet_savedmodel_v2'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name='keras_tensor_1195')
Output Type:
  TensorSpec(shape=(None, 128, 128, 1), dtype=tf.float32, name=None)
Captures:
  1570032208400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1571797271184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1571797277136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1571797270608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1571797277712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1571797270992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1571797270800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1571797277520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1571797271376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1571797278672: TensorSpec(shape=(), dtype=tf.resource, name=None)
